## BONUS: Live Computer Vision with YOLO

Now let's see machine learning in action! We'll use a state-of-the-art YOLO (You Only Look Once) model to detect objects in real-time using your webcam.

**What is YOLO?**
- YOLO is a real-time object detection system
- It can identify and locate multiple objects in images/video
- Used in self-driving cars, security systems, and mobile apps
- Represents the cutting edge of computer vision

**Note:** This section demonstrates advanced ML concepts that we'll cover in Week 4. For now, just enjoy the demonstration!

In [1]:
# Import required libraries for computer vision
import cv2
from ultralytics import YOLO
import time

# Check if webcam is available
def check_webcam():
    """Check if webcam is available"""
    cap = cv2.VideoCapture(0)
    if cap.isOpened():
        print("✓ Webcam detected and ready!")
        cap.release()
        return True
    else:
        print("✗ No webcam detected. Using sample image instead.")
        return False

# Test webcam availability
webcam_available = check_webcam()

print("Setting up YOLO object detection...")
print("This may take a moment to download the model weights...")

# Load pre-trained YOLO model
try:
    # Use YOLOv8 nano model (fastest, good for real-time)
    model_yolo = YOLO('yolov8n.pt')  # This will download the model automatically
    print("✓ YOLO model loaded successfully!")
    
    # Get model information
    print(f"Model: YOLOv8 Nano")
    print(f"Classes: {len(model_yolo.names)} object types can be detected")
    
    # Show some example classes
    print("Example detectable objects:")
    example_classes = ['person', 'car', 'dog', 'cat', 'bottle', 'chair', 'laptop', 'phone']
    for cls in example_classes:
        if cls in model_yolo.names.values():
            print(f"  - {cls.title()}")
    
except Exception as e:
    print(f"Error loading YOLO model: {e}")
    print("Please ensure you have internet connection for model download.")
    model_yolo = None

✓ Webcam detected and ready!
Setting up YOLO object detection...
This may take a moment to download the model weights...
✓ YOLO model loaded successfully!
Model: YOLOv8 Nano
Classes: 80 object types can be detected
Example detectable objects:
  - Person
  - Car
  - Dog
  - Cat
  - Bottle
  - Chair
  - Laptop


In [2]:
# Live webcam object detection function
def run_live_detection(duration=30):
    """
    Run live object detection using webcam
    
    Args:
        duration: How long to run detection (seconds)
    """
    if not webcam_available or model_yolo is None:
        print("Webcam or YOLO model not available.")
        return
    
    print(f"Starting live object detection for {duration} seconds...")
    print("Press 'q' to quit early, or 'c' to capture a frame")
    print("Objects will be detected and labeled in real-time!")
    
    # Initialize webcam
    cap = cv2.VideoCapture(0)
    
    # Set webcam properties for better performance
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    cap.set(cv2.CAP_PROP_FPS, 30)
    
    start_time = time.time()
    frame_count = 0
    
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                print("Failed to capture frame")
                break
            
            # Run YOLO detection
            results = model_yolo(frame, verbose=False)
            
            # Draw detections on frame
            annotated_frame = results[0].plot()
            
            # Add performance info
            frame_count += 1
            fps = frame_count / (time.time() - start_time)
            cv2.putText(annotated_frame, f'FPS: {fps:.1f}', (10, 30), 
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            
            # Display frame
            cv2.imshow('YOLO Live Object Detection', annotated_frame)
            
            # Check for key presses
            key = cv2.waitKey(1) & 0xFF
            if key == ord('q'):
                print("Quit requested by user")
                break
            elif key == ord('c'):
                # Capture frame
                filename = f'captured_frame_{int(time.time())}.jpg'
                cv2.imwrite(filename, annotated_frame)
                print(f"Frame captured: {filename}")
            
            # Check time limit
            if time.time() - start_time > duration:
                print(f"Time limit ({duration}s) reached")
                break
                
    except KeyboardInterrupt:
        print("Detection stopped by user")
    
    finally:
        # Clean up
        cap.release()
        cv2.destroyAllWindows()
        
        # Print summary
        total_time = time.time() - start_time
        avg_fps = frame_count / total_time if total_time > 0 else 0
        print(f"\nDetection Summary:")
        print(f"  Total time: {total_time:.1f} seconds")
        print(f"  Frames processed: {frame_count}")
        print(f"  Average FPS: {avg_fps:.1f}")

# Alternative function for systems without webcam
def run_image_detection():
    """
    Run object detection on a sample image
    """
    if model_yolo is None:
        print("YOLO model not available.")
        return
        
    print("Running YOLO detection on sample image...")
    
    try:
        # Use a sample image for demonstration
        import urllib.request
        sample_image_url = "https://ultralytics.com/images/bus.jpg"
        
        # Download sample image
        urllib.request.urlretrieve(sample_image_url, "sample_image.jpg")
        
        # Run detection
        results = model_yolo("sample_image.jpg")
        
        # Display results
        annotated_image = results[0].plot()
        
        # Convert BGR to RGB for matplotlib
        annotated_image_rgb = cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB)
        
        # Display using matplotlib
        plt.figure(figsize=(12, 8))
        plt.imshow(annotated_image_rgb)
        plt.axis('off')
        plt.title('YOLO Object Detection Results')
        plt.tight_layout()
        plt.show()
        
        # Print detection summary
        print("\nDetection Results:")
        for result in results:
            boxes = result.boxes
            if boxes is not None:
                for box in boxes:
                    class_id = int(box.cls[0])
                    confidence = float(box.conf[0])
                    class_name = model_yolo.names[class_id]
                    print(f"  Detected: {class_name} (confidence: {confidence:.2f})")
        
    except Exception as e:
        print(f"Error in image detection: {e}")
        
print("YOLO functions defined successfully!")

YOLO functions defined successfully!


In [3]:
# Interactive YOLO Detection Demo
print("YOLO Object Detection Demo")
print("=" * 50)

if webcam_available and model_yolo is not None:
    print("Option 1: Live Webcam Detection (Recommended)")
    print("  - Real-time object detection using your webcam")
    print("  - Press 'q' to quit, 'c' to capture frames")
    print("  - Duration: 30 seconds")
    print()
    print("Option 2: Sample Image Detection")
    print("  - Detect objects in a sample image")
    print("  - Good for systems without webcam")
    print()
    
    choice = input("Choose option (1 for webcam, 2 for image, or 'skip'): ").strip()
    
    if choice == '1':
        print("\nStarting live webcam detection...")
        print("Make sure your webcam is not being used by other applications!")
        input("Press Enter when ready...")
        run_live_detection(duration=30)
    elif choice == '2':
        print("\nRunning sample image detection...")
        run_image_detection()
    else:
        print("Skipping YOLO demo.")
else:
    print("No webcam detected or YOLO model failed to load.")
    if model_yolo is not None:
        print("Running sample image detection...")
        run_image_detection()
    else:
        print("Cannot run YOLO demo without model.")

print("\nYOLO Demo Complete!")
print("You just experienced state-of-the-art computer vision!")
print("In Week 4, we'll learn how these models work and build our own.")

YOLO Object Detection Demo
Option 1: Live Webcam Detection (Recommended)
  - Real-time object detection using your webcam
  - Press 'q' to quit, 'c' to capture frames
  - Duration: 30 seconds

Option 2: Sample Image Detection
  - Detect objects in a sample image
  - Good for systems without webcam


Starting live webcam detection...
Make sure your webcam is not being used by other applications!
Starting live object detection for 30 seconds...
Press 'q' to quit early, or 'c' to capture a frame
Objects will be detected and labeled in real-time!
Quit requested by user

Detection Summary:
  Total time: 19.5 seconds
  Frames processed: 294
  Average FPS: 15.0

YOLO Demo Complete!
You just experienced state-of-the-art computer vision!
In Week 4, we'll learn how these models work and build our own.


## What Just Happened? (YOLO Explained Simply)

**The Magic Behind YOLO:**

1. **Neural Network Architecture**: YOLO uses a deep neural network with millions of parameters
2. **Training Data**: Trained on millions of images with labeled objects
3. **Real-time Processing**: Processes entire images in one pass (hence "You Only Look Once")
4. **Bounding Box Regression**: Predicts object locations using coordinate regression
5. **Classification**: Simultaneously classifies what each detected object is

**Why This is Amazing:**
- **Speed**: Can process 30+ frames per second on modern hardware
- **Accuracy**: Achieves over 90% accuracy on common objects
- **Versatility**: Works on any image/video input
- **Real-world Applications**: Self-driving cars, security systems, medical imaging

**Connection to Linear Regression:**
- YOLO uses regression to predict bounding box coordinates (x, y, width, height)
- Multiple regression concepts applied to pixel-level predictions
- Demonstrates how basic ML concepts scale to complex applications

This is where you'll be by the end of the course - building cutting-edge AI systems!